Defining the schema 

In [0]:
%sql
-- creating volume
use catalog ecomm;
use schema landing;

create external volume if not exists operational_data1
location 'abfss://ktdejulproject@azuredatabricksktdejulsa.dfs.core.windows.net/ecomm/landing/operational_data1'

In [0]:
# defining the schema
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType, TimestampType

customers_schema = StructType([
  StructField("customer_id", IntegerType(), True),
  StructField("customer_name", StringType(), True),
  StructField("date_of_birth", DateType(), True),
  StructField("telephone", StringType(), True),
  StructField("email", StringType(), True),
  StructField("member", DateType(), True),
  StructField("created_timestamp", TimestampType(), True)
])

In [0]:
# creating a dataframe
customers_df = (
    spark.readStream
        .format("json")
        .schema(customers_schema)  # Defining schema is best practise for streaming sources
        .load("/Volumes/ecomm/landing/operational_data1/customers_stream/")
)

In [0]:
# basic transformations and stored into another dataframe
from pyspark.sql.functions import col
customer_transform_df = customers_df.withColumn("filepath", col("_metadata"))


In [0]:
# write stream and load into delta table

streaming_query = (customer_transform_df.writeStream
                        .format("delta")
                        .option("checkpointLocation", "/Volumes/ecomm/landing/operational_data1/customers_stream/_checkpoint_stream")
                        .toTable("ecomm.bronze.customers_stream")
                        )

# streaming_query.stop() to stop the streaming process.                         

In [0]:
%sql
select * from ecomm.bronze.customers_stream

customer_id,customer_name,date_of_birth,telephone,email,member,created_timestamp,filepath
9179,Richard Cox,1996-10-25,+1 6680703335,devon84@mail.com,null,2024-10-17T16:12:27Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
4858,Carla Morton,2004-06-21,+1 8616454195,joseph88@mail.com,null,2024-10-01T00:50:29Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
7207,Billy Scott,1997-03-17,+1 5544387564,christopher30@mail.com,null,2024-10-23T22:03:08Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
8539,Lori Mason,2002-11-01,+1 0498301620,stephanie7@mail.com,null,2024-10-12T06:02:27Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
9706,Jennifer Haas,2001-04-03,+1 4725460000,benjamin55@mail.com,null,2024-10-24T13:03:13Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
9263,Joseph Keller,2003-02-11,+1 3817867756,null,null,2024-10-08T22:49:25Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
5028,Jessica Harris,2004-04-19,+1 8604009935,null,null,2024-10-06T19:55:52Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
9018,William Carter,2003-09-05,+1 1448753611,james70@gmail.com,null,2024-10-18T23:24:52Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
8580,Shannon Austin,2002-03-22,+1 4594705629,john30@gmail.com,null,2024-10-21T13:20:26Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"
3409,Andrew Phillips,2003-04-17,+1 4079273853,peter73@yahoo.com,null,2024-10-02T14:53:40Z,"List(dbfs:/Volumes/ecomm/landing/operational_data1/customers_stream/customers_2024_10.json, customers_2024_10.json, 4389, 0, 4389, 2026-09-07T20:23:45Z)"


In [0]:
streaming_query.id

'33c548a1-b2b5-43a8-b08b-a07b4a5e6573'

In [0]:
streaming_query.status

{'message': 'Waiting for data to arrive',
 'isDataAvailable': False,
 'isTriggerActive': False}

Now we will upload new files in that folder. 

In [0]:
%sql
-- to see the history of the delta table

describe history ecomm.bronze.customers_stream

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
2,2026-09-07T20:43:42Z,143296701060486,reshmithau.0424@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 33c548a1-b2b5-43a8-b08b-a07b4a5e6573, epochId -> 1, statsOnLoad -> false)",null,List(2220975625858409),0907-153307-ccxpi65e,1,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 15, numOutputBytes -> 5246, numAddedFiles -> 1)",null,Databricks-Runtime/15.4.x-photon-scala2.12
1,2026-09-07T20:40:57Z,143296701060486,reshmithau.0424@gmail.com,STREAMING UPDATE,"Map(outputMode -> Append, queryId -> 33c548a1-b2b5-43a8-b08b-a07b4a5e6573, epochId -> 0, statsOnLoad -> true)",null,List(2220975625858409),0907-153307-ccxpi65e,0,WriteSerializable,true,"Map(numRemovedFiles -> 0, numOutputRows -> 20, numOutputBytes -> 5491, numAddedFiles -> 1)",null,Databricks-Runtime/15.4.x-photon-scala2.12
0,2026-09-07T20:40:47Z,143296701060486,reshmithau.0424@gmail.com,CREATE TABLE,"Map(partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.enableDeletionVectors"":""true""}, statsOnLoad -> false)",null,List(2220975625858409),0907-153307-ccxpi65e,null,WriteSerializable,true,Map(),null,Databricks-Runtime/15.4.x-photon-scala2.12


In [0]:
# stop streaming

streaming_query.stop()